# Phase 0 — Validation Protocol

**What is this notebook?** Before we build any model, we decide *how we will judge* every model. This notebook creates the data splits that every later phase must respect.

**Why does this come first?** Because every number we report later (AUC, calibration, € savings) is only trustworthy if the data used to *measure* it was never used to *build* anything. If we get this wrong, everything downstream is quietly broken and we won't know it.

**Who cares?**
- *Us* — so we don't fool ourselves with numbers that look better than reality.
- *An interviewer / reviewer* — the first thing a senior ML engineer checks in a project is "how did they validate?"
- *A real bank's model-risk team* — they would reject any model whose evaluation data touched training.

**When?** Once, right now, before any modeling. The splits are then **frozen** — saved to disk with a fixed random seed so they never change.

**Where do results go?** `data/splits/` — three small parquet files that define the split forever.

**How?** Three steps:
1. Cut 20% of applicants into a **holdout** we will not look at until the very end (Phase 7).
2. Split the remaining 80% (**dev set**) into 5 equal **folds** for cross-validation.
3. Run a **pipeline-integrity check**: a model that tries to tell dev rows from holdout rows. If it can (AUC > 0.55), our processing leaked information across the boundary. If it can't (AUC ≈ 0.50), the split is clean.

## 1. Setup

**What:** import our project modules. **Why DRY matters:** all real logic lives in `src/` — this notebook only *calls* it and explains it. If we fix a bug in `src/`, every notebook and script gets the fix automatically.

In [1]:
import sys
sys.path.insert(0, "..")   # so we can import from the project root

import pandas as pd

from config.constants import RANDOM_STATE, N_FOLDS, TEST_SIZE, TARGET_COL
from src.data.load import load_application
from src.validation.protocol import create_holdout_split, get_cv_folds

print(f"Seed = {RANDOM_STATE}, folds = {N_FOLDS}, holdout share = {TEST_SIZE:.0%}")

Seed = 42, folds = 5, holdout share = 20%


## 2. Load the data

**What:** the main application table — one row per loan application.

**Why only this table for splitting?** The split is defined on *applicants* (`SK_ID_CURR`). The auxiliary tables (bureau, previous applications) attach to applicants later, so splitting the main table splits everything.

In [2]:
app = load_application()
print(f"Shape: {app.shape[0]:,} rows x {app.shape[1]} columns")
print(f"Default rate: {app[TARGET_COL].mean():.4f}  (class ratio ~ {(1-app[TARGET_COL].mean())/app[TARGET_COL].mean():.1f}:1)")

Shape: 307,511 rows x 122 columns
Default rate: 0.0807  (class ratio ~ 11.4:1)


## 3. Cut the 20% holdout

**What:** 61,503 applicants set aside, untouched until Phase 7.

**Why "stratified"?** Stratified means both pieces keep the same 8.07% default rate. Without it, random chance could give the holdout 7.5% or 8.6% defaults, and our final numbers would be noisy for a silly reason.

**Why a fixed seed (42)?** So anyone re-running this code gets *exactly* the same split. Reproducibility is not optional in a system meant to be audited.

**How is "don't touch it" enforced?** By habit and by code: every loader function used in development (`load_dev_application`) filters to dev IDs only. The holdout is only readable through a separate, deliberate function.

In [3]:
dev, holdout = create_holdout_split(app)

summary = pd.DataFrame({
    "rows": [len(dev), len(holdout)],
    "default_rate": [dev[TARGET_COL].mean(), holdout[TARGET_COL].mean()],
}, index=["dev (80%)", "holdout (20%)"])
summary

,rows,default_rate
dev (80%),246008,0.080729
holdout (20%),61503,0.080728


## 4. Assign the 5 cross-validation folds

**What:** every dev applicant gets a fold number 0–4.

**Why 5 folds?** Cross-validation lets us evaluate on data the model didn't train on, *without* spending the holdout. Each model trains 5 times, each time leaving one fold out; predictions on the left-out fold are called **out-of-fold (OOF)** predictions.

**Why save the assignment to disk?** Because *every* model (logistic baseline, LightGBM, constrained LightGBM) must use the **same** folds. That makes model comparisons paired — differences come from the models, not from fold luck.

**Why do OOF predictions matter so much?** They are the raw material for Phase 4: calibration is fitted on them, thresholds are tested on them. In-fold (training) predictions are overconfident and would poison both.

In [4]:
import numpy as np

skf = get_cv_folds(dev)
fold_assignment = np.full(len(dev), -1, dtype=int)
for fold_idx, (_, val_idx) in enumerate(skf.split(dev, dev[TARGET_COL])):
    fold_assignment[val_idx] = fold_idx

fold_df = pd.DataFrame({
    "SK_ID_CURR": dev["SK_ID_CURR"].values,
    "fold": fold_assignment,
    TARGET_COL: dev[TARGET_COL].values,
})

# Every fold should have ~49,200 rows and the same 8.07% default rate
fold_df.groupby("fold")[TARGET_COL].agg(rows="count", default_rate="mean")

,rows,default_rate
fold,,
0,49202,0.080728
1,49202,0.080728
2,49202,0.080728
3,49201,0.080730
4,49201,0.080730


## 5. The pipeline-integrity check

**What:** we train a LightGBM whose job is to guess, for each row, "are you from dev or from holdout?" and measure its AUC.

**Why should it fail?** Because the holdout is a *random* slice of the same dataset — there is genuinely nothing that distinguishes it. Expected AUC ≈ 0.50 (coin flip) *by construction*.

**So what's the point?** It's a tripwire for **our own bugs**. If we ever fit an imputer, a WOE binning, or an encoder on data from *both* sides of the split, the processed features will carry a faint fingerprint of the boundary — and this check's AUC will rise above ~0.55. It cannot detect real-world drift (there is none in one Kaggle file); it detects *us* making a mistake.

**When does it run?** Twice: now on raw features (baseline sanity), and again at the end of Phase 2 on the engineered features (where the bugs would actually happen).

**How to read the result:** AUC ≤ 0.55 → PASS. AUC > 0.55 → stop, audit Phase 2 processing.

In [5]:
from src.validation.protocol import pipeline_integrity_check

num_cols = [col for col in app.select_dtypes("number").columns
            if col not in ("SK_ID_CURR", TARGET_COL)]

result = pipeline_integrity_check(dev, holdout, num_cols)
print(f"Mean AUC : {result['mean_auc']:.4f}   (want ~0.50, alarm at >{result['threshold']})")
print(result["interpretation"])

Mean AUC : 0.5009   (want ~0.50, alarm at >0.55)
PASS — no pipeline leakage detected (AUC ≈ 0.50 as expected).


## 6. What we froze today

| Artifact | File | Rule |
|---|---|---|
| Dev IDs (80%) | `data/splits/dev_ids.parquet` | all development happens here |
| Holdout IDs (20%) | `data/splits/holdout_ids.parquet` | **touched once, in Phase 7, by script** |
| Fold assignment | `data/splits/dev_folds.parquet` | every model uses these exact folds |

*(The actual files were written by `scripts/run_phase0.py` — the same functions this notebook demonstrated. Script writes, notebook explains: that's the DRY division of labor.)*

**The leakage-audit rules we carry into Phase 2:**
1. Aggregate auxiliary tables per `SK_ID_CURR` only — the target never crosses a join.
2. Any target encoding is fitted *inside* each CV fold, never on all training data.
3. Imputation statistics and WOE bins are computed on training folds only.

**Checkpoint: PASSED.** Splits are frozen, folds are balanced, integrity AUC ≈ 0.50. Phase 1 may begin.